# CMS Hospital Readmission Risk Analysis
## Notebook 4 — Predictive Modelling

**Input:**  `data/engineered/cms_patients_engineered.csv`  
**Output:** `data/outputs/` — 8 files including scored patients, model metrics, and Power BI tables

---

### What this notebook does
1. Selects the 13 best predictive features
2. Splits data 80/20 (stratified by target)
3. Trains 3 models and evaluates each across 6 metrics
4. Selects Gradient Boosting as champion model
5. Scores all 5,000 patients with readmission probability and risk tier
6. Exports all aggregated tables ready for Power BI

| Model | Role |
|-------|------|
| Logistic Regression | Interpretable baseline |
| Random Forest | Ensemble benchmark |
| Gradient Boosting | Champion — best sensitivity/specificity balance |

### 4.1 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report,
)
from sklearn.calibration import calibration_curve

print('All libraries loaded successfully')
import sklearn; print(f'scikit-learn version: {sklearn.__version__}')

### 4.2 — Load engineered data

In [ ]:
df = pd.read_csv('data/engineered/cms_patients_engineered.csv')

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Target: readmitted_30d')
print(f'Positive class prevalence: {df["readmitted_30d"].mean():.1%}')
print(f'Positive cases: {df["readmitted_30d"].sum():,}')
print(f'Negative cases: {(df["readmitted_30d"]==0).sum():,}')

### 4.3 — Feature selection and preparation

In [ ]:
NUMERIC_FEATURES = [
    'age', 'length_of_stay_days', 'prior_admissions_12mo',
    'comorbidity_index', 'num_medications', 'ed_visits_prior_6mo',
    'bmi', 'social_risk_score',
]
CATEGORICAL_FEATURES = [
    'primary_diagnosis', 'discharge_disposition',
    'payer', 'gender', 'age_group',
]

# Encode categoricals fresh from labels
encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = list(le.classes_)

ENCODED_FEATURES = [f'{c}_enc' for c in CATEGORICAL_FEATURES]
ALL_FEATURES     = NUMERIC_FEATURES + ENCODED_FEATURES

X = df[ALL_FEATURES].fillna(df[ALL_FEATURES].median())
y = df['readmitted_30d']

print(f'Total features: {len(ALL_FEATURES)}')
print(f'  Numeric:    {len(NUMERIC_FEATURES)}')
print(f'  Encoded:    {len(ENCODED_FEATURES)}')
print(f'\nFeature list:')
for i, f in enumerate(ALL_FEATURES, 1):
    print(f'  {i:2}. {f}')

### 4.4 — Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale for logistic regression
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set:   {len(X_train):,} records  |  positive rate: {y_train.mean():.1%}')
print(f'Test set:       {len(X_test):,} records   |  positive rate: {y_test.mean():.1%}')
print(f'Split ratio:    80% train / 20% test (stratified)')

### 4.5 — Train and evaluate all 3 models

In [ ]:
model_configs = {
    'Logistic Regression': {
        'model':  LogisticRegression(C=0.5, max_iter=500, random_state=42),
        'scaled': True,
    },
    'Random Forest': {
        'model':  RandomForestClassifier(
            n_estimators=150, max_depth=6, min_samples_leaf=20,
            random_state=42, n_jobs=-1),
        'scaled': False,
    },
    'Gradient Boosting': {
        'model':  GradientBoostingClassifier(
            n_estimators=150, max_depth=4, learning_rate=0.08,
            subsample=0.85, min_samples_leaf=20, random_state=42),
        'scaled': False,
    },
}

results = {}
print('=' * 60)
print('  MODEL TRAINING & EVALUATION')
print('=' * 60)

for name, cfg in model_configs.items():
    print(f'\n  Training: {name}...')
    mdl = cfg['model']
    Xtr = X_train_sc if cfg['scaled'] else X_train
    Xte = X_test_sc  if cfg['scaled'] else X_test

    mdl.fit(Xtr, y_train)
    proba = mdl.predict_proba(Xte)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    auc    = roc_auc_score(y_test, proba)
    apr    = average_precision_score(y_test, proba)
    brier  = brier_score_loss(y_test, proba)
    cv_auc = cross_val_score(
        mdl, Xtr, y_train, cv=StratifiedKFold(5),
        scoring='roc_auc', n_jobs=-1).mean()

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    results[name] = {
        'AUC_ROC':      round(float(auc),    4),
        'PR_AUC':       round(float(apr),    4),
        'Brier_Score':  round(float(brier),  4),
        'CV_AUC_5fold': round(float(cv_auc), 4),
        'Sensitivity':  round(float(tp/(tp+fn)), 4),
        'Specificity':  round(float(tn/(tn+fp)), 4),
        'PPV':          round(float(tp/(tp+fp)) if (tp+fp)>0 else 0, 4),
        'NPV':          round(float(tn/(tn+fn)) if (tn+fn)>0 else 0, 4),
        'TP':int(tp),'FP':int(fp),'TN':int(tn),'FN':int(fn),
    }
    r = results[name]
    print(f'    AUC-ROC:         {r["AUC_ROC"]:.4f}')
    print(f'    PR-AUC:          {r["PR_AUC"]:.4f}')
    print(f'    Brier Score:     {r["Brier_Score"]:.4f}  (lower = better calibration)')
    print(f'    CV AUC (5-fold): {r["CV_AUC_5fold"]:.4f}')
    print(f'    Sensitivity:     {r["Sensitivity"]:.1%}  |  Specificity: {r["Specificity"]:.1%}')
    print(f'    PPV:             {r["PPV"]:.1%}  |  NPV: {r["NPV"]:.1%}')
    print(f'    Confusion:       TP={r["TP"]}  FP={r["FP"]}  TN={r["TN"]}  FN={r["FN"]}')

### 4.6 — Model comparison table

In [ ]:
comparison = pd.DataFrame(results).T[['AUC_ROC','PR_AUC','Brier_Score','CV_AUC_5fold','Sensitivity','Specificity','PPV','NPV']]
comparison.index.name = 'Model'
print('Model comparison (champion marked with ★):')
print(comparison.to_string())

### 4.7 — Champion model: Gradient Boosting — feature importance

In [ ]:
champion = model_configs['Gradient Boosting']['model']

FEATURE_LABELS = {
    'social_risk_score':         'Social risk score',
    'length_of_stay_days':       'Length of stay',
    'comorbidity_index':         'Comorbidity index',
    'age':                       'Patient age',
    'bmi':                       'BMI',
    'prior_admissions_12mo':     'Prior admissions (12mo)',
    'num_medications':           'Medication count',
    'ed_visits_prior_6mo':       'ED visits (6mo)',
    'discharge_disposition_enc': 'Discharge disposition',
    'primary_diagnosis_enc':     'Primary diagnosis',
    'payer_enc':                 'Payer type',
    'gender_enc':                'Gender',
    'age_group_enc':             'Age group',
}

fi = pd.DataFrame({
    'feature':    ALL_FEATURES,
    'importance': champion.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)
fi['rank']           = fi.index + 1
fi['importance_pct'] = (fi['importance'] / fi['importance'].sum() * 100).round(2)
fi['feature_label']  = fi['feature'].map(FEATURE_LABELS).fillna(fi['feature'])

print('Feature importance — Gradient Boosting champion:')
print()
for _, row in fi.iterrows():
    bar = '█' * int(row['importance_pct'] / 1.2)
    print(f'  {int(row["rank"]):2}. {row["feature_label"]:<28}  {row["importance_pct"]:5.1f}%  {bar}')

### 4.8 — Score all 5,000 patients

In [ ]:
X_full     = df[ALL_FEATURES].fillna(df[ALL_FEATURES].median())
full_proba = champion.predict_proba(X_full)[:, 1]

df['model_readmit_prob'] = full_proba.round(4)
df['model_risk_score']   = (full_proba * 100).round(1)

def assign_tier(p):
    if p >= 0.60: return 'Critical'
    if p >= 0.40: return 'High'
    if p >= 0.20: return 'Moderate'
    return 'Low'

df['model_risk_tier'] = df['model_readmit_prob'].apply(assign_tier)

print('Risk tier distribution (model output):')
print()
for tier in ['Critical','High','Moderate','Low']:
    n   = (df['model_risk_tier'] == tier).sum()
    pct = n / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {tier:<10}  {n:>5,} ({pct:.1f}%)  {bar}')

### 4.9 — Build Power BI aggregation tables

In [ ]:
os.makedirs('data/outputs', exist_ok=True)

# Monthly KPIs
df['admit_date_dt'] = pd.to_datetime(df['admit_date'])
df['month_label']   = df['admit_date_dt'].dt.to_period('M').astype(str)
monthly = df.groupby('month_label').agg(
    admissions    =('patient_id','count'),
    readmissions  =('readmitted_30d','sum'),
    avg_risk_score=('model_risk_score','mean'),
    total_cost    =('total_cost_usd','sum'),
    avg_cost      =('total_cost_usd','mean'),
    avg_los       =('length_of_stay_days','mean'),
).reset_index()
monthly['readmission_rate_pct'] = (monthly['readmissions'] / monthly['admissions'] * 100).round(2)

# Diagnosis summary
dx = df.groupby('primary_diagnosis').agg(
    patients     =('patient_id','count'),
    readmissions =('readmitted_30d','sum'),
    avg_risk_score=('model_risk_score','mean'),
    avg_cost     =('total_cost_usd','mean'),
    total_cost   =('total_cost_usd','sum'),
).reset_index()
dx['readmission_rate_pct'] = (dx['readmissions'] / dx['patients'] * 100).round(2)
dx = dx.sort_values('readmission_rate_pct', ascending=False)

# Hospital summary
hosp = df.groupby('hospital').agg(
    patients     =('patient_id','count'),
    readmissions =('readmitted_30d','sum'),
    avg_risk_score=('model_risk_score','mean'),
    avg_cost     =('total_cost_usd','mean'),
    total_cost   =('total_cost_usd','sum'),
).reset_index()
hosp['readmission_rate_pct'] = (hosp['readmissions'] / hosp['patients'] * 100).round(2)

# Risk tier summary
tiers = df.groupby('model_risk_tier').agg(
    patients     =('patient_id','count'),
    readmissions =('readmitted_30d','sum'),
    avg_cost     =('total_cost_usd','mean'),
    total_cost   =('total_cost_usd','sum'),
    avg_risk_score=('model_risk_score','mean'),
).reset_index()
tiers['readmission_rate_pct'] = (tiers['readmissions'] / tiers['patients'] * 100).round(2)

# Patient watchlist — top 500 Critical patients
watchlist = df[df['model_risk_tier']=='Critical'].nlargest(500,'model_risk_score')[[
    'patient_id','age','gender','primary_diagnosis','hospital','payer',
    'discharge_disposition','length_of_stay_days','prior_admissions_12mo',
    'comorbidity_index','num_medications','composite_risk_flags',
    'model_risk_score','model_risk_tier','total_cost_usd',
]].copy()

# Calibration
champ_proba_test = champion.predict_proba(X_test)[:, 1]
frac_pos, mean_pred = calibration_curve(y_test, champ_proba_test, n_bins=10)
cal_df = pd.DataFrame({
    'mean_predicted_prob': mean_pred.round(3),
    'fraction_positives':  frac_pos.round(3),
    'calibration_error':   (mean_pred - frac_pos).round(3),
})

print('Aggregation tables built:')
print(f'  Monthly KPIs:     {len(monthly)} months')
print(f'  Diagnosis:        {len(dx)} diagnoses')
print(f'  Hospital:         {len(hosp)} hospitals')
print(f'  Risk tiers:       {len(tiers)} tiers')
print(f'  Patient watchlist: {len(watchlist):,} critical patients')

### 4.10 — Save all outputs

In [ ]:
# Remove temp columns before saving
df.drop(columns=['admit_date_dt','month_label'], errors='ignore', inplace=True)

df.to_csv('data/outputs/fact_patients_scored.csv', index=False)
fi.to_csv('data/outputs/feature_importance.csv', index=False)
cal_df.to_csv('data/outputs/model_calibration.csv', index=False)
monthly.to_csv('data/outputs/agg_monthly_kpis.csv', index=False)
dx.to_csv('data/outputs/agg_diagnosis.csv', index=False)
hosp.to_csv('data/outputs/agg_hospital.csv', index=False)
tiers.to_csv('data/outputs/agg_risk_tiers.csv', index=False)
watchlist.to_csv('data/outputs/patient_watchlist.csv', index=False)

os.makedirs('reports', exist_ok=True)
with open('reports/model_performance.json', 'w') as f:
    json.dump(results, f, indent=2)

print('All outputs saved to data/outputs/:')
print(f'  fact_patients_scored.csv     — {len(df):,} rows, {df.shape[1]} cols')
print(f'  feature_importance.csv       — {len(fi)} features ranked')
print(f'  model_calibration.csv        — {len(cal_df)} bins')
print(f'  agg_monthly_kpis.csv         — {len(monthly)} months')
print(f'  agg_diagnosis.csv            — {len(dx)} diagnoses')
print(f'  agg_hospital.csv             — {len(hosp)} hospitals')
print(f'  agg_risk_tiers.csv           — {len(tiers)} tiers')
print(f'  patient_watchlist.csv        — {len(watchlist):,} patients')
print(f'  reports/model_performance.json')

### 4.11 — Final summary

In [ ]:
print('=' * 60)
print('  PIPELINE COMPLETE — FINAL SUMMARY')
print('=' * 60)
print(f'  Total patients scored:     {len(df):,}')
print(f'  Readmission rate:          {df["readmitted_30d"].mean():.1%}')
print(f'  Champion model:            Gradient Boosting')
print(f'  Model AUC-ROC:             {results["Gradient Boosting"]["AUC_ROC"]:.4f}')
print(f'  Model Sensitivity:         {results["Gradient Boosting"]["Sensitivity"]:.1%}')
print(f'  Model Specificity:         {results["Gradient Boosting"]["Specificity"]:.1%}')
print(f'  Critical-risk patients:    {(df["model_risk_tier"]=="Critical").sum():,}')
print(f'  Total dataset cost:        ${df["total_cost_usd"].sum()/1e6:.1f}M')
print('=' * 60)
print()
print('All outputs are in data/outputs/ — ready for Power BI import.')

---
**Notebook 4 complete — full pipeline finished.**

Import the files from `data/outputs/` into Power BI Desktop to build the dashboard.